In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os

In [ ]:
load_dotenv(override=True)

True

#### **with: we use to ensure opened file or server will be closed no matter what**
#### **`async` and `await` always come in pairs. `async` defines a function that can wait, `await` is where it actually waits. In Jupyter you can skip `async def` because Jupyter handles it automatically.**

## **01. FIRST MCP SERVER FETCH**
Note: Fetches the raw content of any URL and returns it as text. Used when agent needs to read a webpage.

In [ ]:

fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.list_tools()

fetch_tools

[Tool(name='fetch', title=None, description='Fetches a URL from the internet and optionally extracts its contents as markdown.\n\nAlthough originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.', inputSchema={'description': 'Parameters for fetching a URL.', 'properties': {'url': {'description': 'URL to fetch', 'format': 'uri', 'minLength': 1, 'title': 'Url', 'type': 'string'}, 'max_length': {'default': 5000, 'description': 'Maximum number of characters to return.', 'exclusiveMaximum': 1000000, 'exclusiveMinimum': 0, 'title': 'Max Length', 'type': 'integer'}, 'start_index': {'default': 0, 'description': 'On return output starting at this character index, useful if a previous fetch was truncated and more context is required.', 'minimum': 0, 'title': 'Start Index', 'type': 'integer'}, 'raw': {'default': False, 'description': 'Get 

## **02. NEXT MCP SERVER FETCH**
Note: Controls a real browser. Used when your agent needs to interact with a webpage click buttons, fill forms, scroll, take screenshots. Useful for sites that don't work with simple fetching.

In [ ]:

playwright_params = {"command": "npx","args": [ "@playwright/mcp@latest"]}

async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as server:
    playwright_tools = await server.list_tools()

playwright_tools


[Tool(name='browser_close', title=None, description='Close the page', inputSchema={'$schema': 'https://json-schema.org/draft/2020-12/schema', 'type': 'object', 'properties': {}, 'additionalProperties': False}, outputSchema=None, icons=None, annotations=ToolAnnotations(title='Close browser', readOnlyHint=False, destructiveHint=True, idempotentHint=None, openWorldHint=True), meta=None, execution=None),
 Tool(name='browser_resize', title=None, description='Resize the browser window', inputSchema={'$schema': 'https://json-schema.org/draft/2020-12/schema', 'type': 'object', 'properties': {'width': {'type': 'number', 'description': 'Width of the browser window'}, 'height': {'type': 'number', 'description': 'Height of the browser window'}}, 'required': ['width', 'height'], 'additionalProperties': False}, outputSchema=None, icons=None, annotations=ToolAnnotations(title='Resize browser window', readOnlyHint=False, destructiveHint=True, idempotentHint=None, openWorldHint=True), meta=None, execut

## **03. NEXT MCP SERVER FETCH**
Note: Gives the agent ability to read and write files inside a specific folder. Always create the sandbox folder before starting this server.

In [ ]:
mkdir -p /mnt/d/study_2/MCP_Project/sandbox

In [ ]:
sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
files_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}

async with MCPServerStdio(params=files_params,client_session_timeout_seconds=60) as server:
    file_tools = await server.list_tools()

file_tools

[Tool(name='read_file', title='Read File (Deprecated)', description='Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'path': {'type': 'string'}, 'tail': {'description': 'If provided, returns only the last N lines of the file', 'type': 'number'}, 'head': {'description': 'If provided, returns only the first N lines of the file', 'type': 'number'}}, 'required': ['path']}, outputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'content': {'type': 'string'}}, 'required': ['content'], 'additionalProperties': False}, icons=None, annotations=ToolAnnotations(title=None, readOnlyHint=True, destructiveHint=None, idempotentHint=None, openWorldHint=None), meta=None, execution=ToolExecution(taskSupport='forbidden')),
 Tool(name='read_text_file', title='Read Text File', description="Read the complete contents of a fil

### **now.. bring on the Agent with Tools!**

In [ ]:
company_name = input("Enter company name: ").strip()

if not company_name:
    raise ValueError("Company name cannot be empty")

instructions = """
You browse the internet to accomplish your instructions.
You are highly capable at browsing the internet independently to accomplish your task, 
including accepting all cookies and clicking 'not now' as appropriate to get to the content you need. 
If one website isn't fruitful, try another. Be persistent until you have solved your assignment,
trying different options and sites as needed.
When you need to write files, you do that inside the sandbox folder only.
Only include data you actually found online — never invent numbers.
If data is unavailable write "Data not publicly available".
Always mention your sources at the bottom of the report.
"""

task = f"""
Research {company_name} thoroughly by browsing the internet across multiple sources 
including their official website, Wikipedia, news articles, Crunchbase, LinkedIn, and financial sites.

Then create a single professional HTML report and save it as {company_name.lower().replace(' ', '_')}_report.html in the sandbox folder.

REPORT STRUCTURE:
1. Header — company logo URL if found, company name, tagline, founding year, headquarters
2. Executive Summary — 2-3 paragraph overview
3. Business Model — how they make money, revenue streams
4. Key Metrics — ONLY if you find real numbers:
   - Revenue figures (use a bar chart showing year over year if multiple years found)
   - Employee count over time (line chart only if multiple data points found)
   - Funding rounds with amounts (bar chart showing each round)
   - Market share % vs competitors (pie chart only if real % data found)
4. Leadership Team — name, title, brief background in a clean card layout
5. Products & Services — what they offer, with descriptions
6. Competitors — comparison table with 3-5 competitors
7. Recent News — last 5-10 news headlines with dates and sources
8. SWOT Analysis — ALWAYS as a 2x2 styled grid, NEVER as a chart
9. Future Outlook — key opportunities and risks in paragraph form
10. Sources — list every website you visited

CHART RULES — VERY IMPORTANT:
- ONLY create a chart when you have REAL numerical data with at least 2 data points
- Revenue chart: only if you found revenue figures for 2+ years
- Funding chart: only if you found multiple funding rounds with amounts
- Employee chart: only if you found headcount for 2+ time periods
- Market share: only if you found actual percentage data
- NEVER create a chart for SWOT, leadership, products, or qualitative information
- If you don't have enough data for a chart, use a table or text instead

HTML DESIGN REQUIREMENTS:
- Use Chart.js from CDN for all charts: <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
- Dark professional theme — background #0f1117, cards #1e2130, accent color #4f8ef7
- Clean sans-serif font — use Inter from Google Fonts
- Responsive layout with proper padding and margins
- Each section in its own card with subtle border and shadow
- Competitor comparison as a proper styled HTML table
- SWOT as a 2x2 CSS grid with colored quadrants:
  Strengths = green, Weaknesses = red, Opportunities = blue, Threats = orange
- Leadership team as horizontal cards with name and title
- Color coded news section with date badges
- Professional footer with sources listed cleanly
- The report should look like something from McKinsey or BCG
"""

async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as mcp_server_files:
    async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as mcp_server_browser:
        agent = Agent(
            name="research_analyst",
            instructions=instructions,
            model="gpt-4o",
            mcp_servers=[mcp_server_files, mcp_server_browser]
        )
        try:
            with trace("company_research"):
                result = await Runner.run(
                    agent,
                    task,
                    max_turns=50
                )
                print(result.final_output)
        except Exception as e:
            print(f"Research failed: {e}")

I have created the report on Enpal and saved it as `enpal_report.html`. If you need further information, please let me know!


Notes: 
1. **Agent: OpenAI Agents SDK class. Bundles the model, instructions, and MCP servers together into one agent.**
2. **Runner.run(): Executes the agent in a loop. Sends task to LLM, LLM decides which tool to call, Runner calls it via MCP, result goes back to LLM. Repeats until done.**
3. **trace(): Records every step of the agent run for debugging. Viewable at platform.openai.com/traces.**
4. **Nested async with: Used to keep multiple MCP servers running at the same time. Both servers stay alive for the duration of the nested block.**

### **To keep Track of the Traces**

https://platform.openai.com/traces



### **Some MCP marketplaces**

1. https://mcp.so

2. https://glama.ai/mcp

**Some material to read**

3. https://smithery.ai/

4. https://huggingface.co/blog/LLMhacker/top-11-essential-mcp-libraries

5. HuggingFace great community article: https://huggingface.co/blog/Kseniase/mcp



